# Generation and evaluation (debug)

Goal:
- reuse the retrieval pipeline built earlier
- retrieve top-k chunks for each question
- build a prompt from the retrieved context
- generate an answer with Llama 3.2 1B
- evaluate predictions with F1, EM, Recall@k, and latency

Important:
- this notebook is for debugging and validating the full pipeline on a small subset first (Testing on CPU directly from the notebook)
- the final large scale experiment will later be moved to a Python script and run with Slurm on GPU (To have access to GPUs with LMU servers are only via slurm)
- retrieval settings to evaluate later:
  - chunk_size in {32, 128, 256}
  - top_k in {1, 5, 10}

In [104]:
# Imports for data loading, indexing, generation, and evaluation

import json
import time
import re
import string
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [105]:

data_dir = Path("/home/a/arfaoui/rag_project/data")

# Retrieval settings that will later be varied systematically
chunk_sizes = [32, 128, 256]
top_k_values = [1, 5, 10]

# For testing, we start with one configuration
debug_chunk_size = 128
debug_top_k = 5

# Start with a small number of questions for fast testing
debug_n_questions = 5
# print to check
print("Data directory:", data_dir)
print("Test chunk size:", debug_chunk_size)
print("Test top-k:", debug_top_k)
print("Test number of questions:", debug_n_questions)

Data directory: /home/a/arfaoui/rag_project/data
Test chunk size: 128
Test top-k: 5
Test number of questions: 5


In [106]:
# Load saved embeddings and metadata for all chunk sizes
# We need these to rebuild or reuse the retrieval pipeline inside this notebook 

all_embeddings = {}
all_metadata = {}

for size in chunk_sizes:
    # Load embedding matrix
    emb_path = data_dir / f"embeddings_{size}.npy"
    embeddings = np.load(emb_path)
    all_embeddings[size] = embeddings

    # Load chunk metadata
    meta_path = data_dir / f"chunks_metadata_{size}.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    all_metadata[size] = metadata

    print(f"\nchunk_size={size}")
    print("Embeddings shape:", embeddings.shape)
    print("Metadata entries:", len(metadata))


chunk_size=32
Embeddings shape: (22186, 384)
Metadata entries: 22186

chunk_size=128
Embeddings shape: (7295, 384)
Metadata entries: 7295

chunk_size=256
Embeddings shape: (5236, 384)
Metadata entries: 5236


In [107]:
# Load precomputed FAISS indexes from disk

faiss_indexes = {}

for size in chunk_sizes:
    index_path = data_dir / f"faiss_index_{size}.index"
    index = faiss.read_index(str(index_path))
    
    faiss_indexes[size] = index
    print(f"Loaded FAISS index for chunk_size={size}")
    print("Index size:", index.ntotal)

Loaded FAISS index for chunk_size=32
Index size: 22186
Loaded FAISS index for chunk_size=128
Index size: 7295
Loaded FAISS index for chunk_size=256
Index size: 5236


In [ ]:
# Load the fixed HotpotQA sample from disk
# The sample file is in JSON Lines format, so we use the datasets library.

sample_path = data_dir / "hotpotqa_sample_500.json"
hotpot_sample = load_dataset("json", data_files=str(sample_path))["train"]
# print to check 
print("Loaded questions:", len(hotpot_sample))
print("Example question:", hotpot_sample[0]["question"])
print("Example answer:", hotpot_sample[0]["answer"])
# Create a small testing subset of questions
# We use only the first N questions for quick pipeline testing.

debug_sample = hotpot_sample.select(range(debug_n_questions))

print("Debug subset size:", len(debug_sample))
print("First debug question:", debug_sample[0]["question"])

In [ ]:
# Load the same embedding model used earlier for chunk embeddings
# We need it here to embed questions for retrieval.

embedding_model_name = "BAAI/bge-small-en-v1.5"
embed_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:", embedding_model_name)

In [ ]:
# Same function from notebook 05 to retrieve top-k chunks for a given question

def retrieve_top_k(question, index, metadata, k=5):
    """
    Given a question, retrieve top-k most similar chunks.
    
    Returns:
    - retrieved_chunks: list of metadata entries
    - scores: similarity scores
    """
    
    # BGE models expect "query: " prefix for queries
    query_text = "query: " + question
    
    # Encode query
    query_embedding = embed_model.encode([query_text])
    
    # Convert to float32 (FAISS requirement)
    query_embedding = np.array(query_embedding).astype("float32")
    
    # Search in FAISS index
    D, I = index.search(query_embedding, k)
    
    # Retrieve corresponding chunks
    retrieved_chunks = [metadata[idx] for idx in I[0]]
    scores = D[0]
    
    return retrieved_chunks, scores

In [ ]:
# Build a single text context string from retrieved chunks
# This will later be passed into the LLM prompt, without it the retrieved chunks are useful information but can't give it to the model
# This makes it:
#  -human-readable
#  -structured for the LLM
#  -easy to debug


def build_retrieved_context(retrieved_chunks):
    """
    Concatenate retrieved chunks into one context string.
    """
    parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        parts.append(f"[Chunk {i}] Title: {chunk['title']}\n{chunk['text']}")

    return "\n\n".join(parts)

## LLM generation setup

In this section, we load the fixed generation model and define a prompt template.

Important:
- the LLM is fixed across all experiments
- the prompt must remain constant across all configurations
- the model should answer using only the retrieved context
- this stage is still for testing on a small subset before the final Slurm run  
  
## Prompt design

We use one fixed prompt template across all experiments.

Design goals:
- force the model to rely only on retrieved context
- keep the prompt simple and constant

In [ ]:
# Load the generation model and tokenizer
# We keep the model fixed across all experiments so that we isolate the effect of the chunk size and top-k 

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)

# Load causal language model
# device_map="auto" lets transformers place the model on available hardware
# torch_dtype="auto" uses a suitable dtype automatically when possible
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)
# print to check 
print("LLM tokenizer loaded:", llm_name)
print("LLM model loaded:", llm_name)

In [ ]:
# Build a fixed prompt from retrieved context and question
# Avoid the model’s external world knowledge

def build_prompt(question, context):
    """
    Create the final prompt shown to the LLM.

    Parameters
    ----------
    question : str
        Input question
    context : str
        Retrieved context built from top-k chunks

    Returns
    -------
    prompt : str
        Final prompt string for generation
    """

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not contained in the context, say "not found".

Context:
{context}

Question:
{question}

Answer:"""

    return prompt

In [ ]:
# Test retrieval + context building + prompt construction on one example question

example = debug_sample[0]

question = example["question"]
correct_answer = example["answer"]

# Use the balanced debug setting first
index = faiss_indexes[debug_chunk_size]
metadata = all_metadata[debug_chunk_size]

retrieved_chunks, retrieval_scores = retrieve_top_k(
    question=question,
    index=index,
    metadata=metadata,
    k=debug_top_k
)

context = build_retrieved_context(retrieved_chunks)
prompt = build_prompt(question, context)

print("Question:", question)
print("correct ansewer:", correct_answer)
print("\n--- Prompt preview ---\n")
print(prompt[:2000])  # preview first part only

In [ ]:
# Generate an answer from the LLM given a prompt
# We keep generation deterministic for fair comparison across experiments.

def generate_answer(prompt, tokenizer, model, max_new_tokens=64):
    """
    Generate an answer from the LLM.

    Parameters
    ----------
    prompt : str
        Input prompt
    tokenizer : tokenizer
    model : causal LM
    max_new_tokens : int
        Maximum number of tokens to generate

    Returns
    -------
    generated_text : str
        Model output text
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # deterministic
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # after testing I recognized that the llm generates the full prompt + context + question and then added answer but we only need answer for the right evaluation 
    # Extract only the answer part
    # look for answer in the generated text 
    if "Answer:" in generated_text:
        # Return everything after "Answer" 
        answer = generated_text.split("Answer:")[-1].strip()
    else:
        # fallback (in case format changes)
        answer = generated_text.strip()

    # remove trailing artifacts (optional cleanup)
    answer = answer.split("\n")[0].strip()

    return answer

In [ ]:
# Run one full pipeline example: retrieval → prompt → generation

start_time = time.time()

generated_answer = generate_answer(
    prompt,
    tokenizer=llm_tokenizer,
    model=llm_model
)

latency = time.time() - start_time

print("=== MODEL OUTPUT ===\n")
print(generated_answer)

print("\n=== LATENCY ===")
print(f"{latency:.4f} seconds")

# Observation
The prediction of the LLM is  the right answer as we can see but the formulation can lead to wrong metrics and effect the EM HOW?:
  - the predicted answer contain for example "." but the truth from data sample doesn't contain "."  which can break the exact EM  
**How to slove this**  
Normalize the text:
  - Lowercase
  - remove punctuation
  - ....   
**With text normalization predection and answer should match** 

In [ ]:
import string

def normalize_text(text):
    """
    Normalize text for fair comparison.

    - lowercase
    - remove punctuation
    - strip whitespace
    """

    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()

    return text

In [ ]:
norm_pred = normalize_text(generated_answer )
norm_gt = normalize_text(correct_answer)

print("Normalized prediction:", norm_pred)
print("Normalized correct_answer:", norm_gt)

In [ ]:
# Evaluation functions

# Exact Match after normalization
# Returns 1 if prediction and correct answer match exactly, else 0.

def compute_em(prediction, correct_answer):
    pred_norm = normalize_text(prediction)
    gt_norm = normalize_text(correct_answer)
    return int(pred_norm == gt_norm)
    
# Token-level F1 score
# This measures overlap between predicted answer tokens and correct answer tokens.

def compute_f1(prediction, correct_answer):
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(correct_answer).split()

    # Handle empty cases safely
    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    # Count token overlap
    common_tokens = set(pred_tokens) & set(gt_tokens)
    num_same = sum(min(pred_tokens.count(tok), gt_tokens.count(tok)) for tok in common_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return f1
    
# Recall@k for retrieval
# Returns 1 if the answer appears in at least one retrieved chunk, else 0.

def compute_recall_at_k(retrieved_chunks, correct_answer):
    gt_norm = normalize_text(correct_answer)

    for chunk in retrieved_chunks:
        chunk_text_norm = normalize_text(chunk["text"])
        if gt_norm in chunk_text_norm:
            return 1

    return 0

In [ ]:
# Test metrics on the current single example

em = compute_em(generated_answer, correct_answer)
f1 = compute_f1(generated_answer, correct_answer)
recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)

print("Generated answer:", generated_answer)
print("Correct answer:", correct_answer)
print("EM:", em)
print("F1:", f1)
print("Recall@k:", recall_k)

In [ ]:
def run_single_example(example, chunk_size, top_k):
    """
    Run full RAG pipeline for one example.

    Returns a dictionary with:
    - prediction
    - correct answer
    - EM
    - F1
    - Recall@k
    - latency
    """

    question = example["question"]
    correct_answer = example["answer"]

    # Select correct index + metadata for this chunk size
    index = faiss_indexes[chunk_size]
    metadata = all_metadata[chunk_size]

    # --- Retrieval ---
    retrieved_chunks, _ = retrieve_top_k(
        question=question,
        index=index,
        metadata=metadata,
        k=top_k
    )

    # --- Context building ---
    context = build_retrieved_context(retrieved_chunks)

    # --- Prompt ---
    prompt = build_prompt(question, context)

    # --- Generation + latency ---
    start_time = time.time()

    prediction = generate_answer(
        prompt,
        tokenizer=llm_tokenizer,
        model=llm_model
    )

    latency = time.time() - start_time

    # --- Metrics ---
    em = compute_em(prediction, correct_answer)
    f1 = compute_f1(prediction, correct_answer)
    recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)

    return {
        "question": question,
        "prediction": prediction,
        "correct_answer": correct_answer,
        "EM": em,
        "F1": f1,
        "Recall@k": recall_k,
        "latency": latency
    }

In [ ]:
def run_test_evaluation(debug_sample, chunk_size, top_k):
    """
    Run evaluation on a small subset of questions.
    """

    results = []

    for i, example in enumerate(debug_sample):
        print(f"Running example {i+1}/{len(debug_sample)}")

        result = run_single_example(
            example=example,
            chunk_size=chunk_size,
            top_k=top_k
        )

        results.append(result)

    return results

In [ ]:
results = run_test_evaluation(
    debug_sample=debug_sample,
    chunk_size=debug_chunk_size,
    top_k=debug_top_k
)